#Trabalho de Inteligência Artificial

Dataset utilizado: KDDCUP99  
O conjunto de dados contém uma variedade de intrusões de rede.  
São 43 features e 39 tipos de ataques no dataset.  

Etapas:

1. Utilizou-se o dataset KDDCUP99 com uma pequena amostra de dados de treino e teste
2. Limpeza de dados para organizar o dataset
3. Pré-processamento de dados para preparar dados em categóricos
4. Analise dos melhores algoritmos para dados desbalanceados
5. Matriz de Confusão para extração das melhores features (técnica para redução de dataset)
6. Escolha dos algoritmos Naive Bayes, Random Forest, KNN e Decision Tree e utilização do GridSearch para escolher os melhores parametros (hiperparametrização)
7. Treinamento com 20% de amostras de teste e avaliação de Acurácia, Precisão, Recall e F1-Score para cada algoritmo por classe
8. Treinamento de uma rede neural com Tensorflow e avaliação das mesmas métricas
9. Balanceamento das classes com método SMOTE
10. Utilização do modelo com menor acurácia e da rede neural para verificar como o balanceamento funciona nesses algoritmos e concluir os resultados obtidos.


#### Importação das bibliotecas e base de dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')

from google.colab import drive
drive.mount('/content/drive/')

train = pd.read_csv('/content/drive/MyDrive/trabalho_ia/data_treino.csv')
test = pd.read_csv('/content/drive/MyDrive/trabalho_ia/data_teste.csv')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


#### Concatenando as bases de dados de treino e teste e limpando os dados (features)

In [ ]:
# nomeando as colunas com o nome de cada atributo, concatenando os dados de treino e teste e dropando a coluna difficulty_level
train.shape
test.shape
train.columns = range(train.shape[1])
test.columns = range(test.shape[1])
labels = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
'num_access_files', 'num_outbound_cmds', 'is_host_login',
'is_guest_login', 'count', 'srv_count', 'serror_rate',
'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
'dst_host_srv_rerror_rate', 'attack_type', 'difficulty_level']
combined_data = pd.concat([train, test])
combined_data.shape
combined_data.head(5)

combined_data.columns = labels
combined_data = combined_data.drop('difficulty_level', 1)
combined_data.head(3)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack_type
0,0,udp,other,SF,146,0,0,0,0,0,...,1,0.0,0.60,0.88,0.00,0.00,0.00,0.0,0.00,normal
1,0,tcp,private,S0,0,0,0,0,0,0,...,26,0.1,0.05,0.00,0.00,1.00,1.00,0.0,0.00,neptune
2,0,tcp,http,SF,232,8153,0,0,0,0,...,255,1.0,0.00,0.03,0.04,0.03,0.01,0.0,0.01,normal


#### Pré-processamento dos dados

In [ ]:
from sklearn import preprocessing

# transformando os dados não-categóricos com LabelEncoder
print(set(list(combined_data['attack_type'])))
combined_data['attack_type'] = preprocessing.LabelEncoder().fit_transform(combined_data['attack_type'])
combined_data['protocol_type'] = preprocessing.LabelEncoder().fit_transform(combined_data['protocol_type'])
combined_data['service'] = preprocessing.LabelEncoder().fit_transform(combined_data['service'])
combined_data['flag'] = preprocessing.LabelEncoder().fit_transform(combined_data['flag'])

{'satan', 'portsweep', 'back', 'apache2', 'multihop', 'ipsweep', 'perl', 'ps', 'mscan', 'named', 'nmap', 'land', 'pod', 'processtable', 'snmpguess', 'neptune', 'normal', 'sendmail', 'xsnoop', 'saint', 'xlock', 'sqlattack', 'loadmodule', 'httptunnel', 'spy', 'rootkit', 'worm', 'warezclient', 'phf', 'ftp_write', 'mailbomb', 'udpstorm', 'teardrop', 'imap', 'smurf', 'buffer_overflow', 'xterm', 'guess_passwd', 'snmpgetattack', 'warezmaster'}


In [ ]:
# verificar a quantidade de amostras por ataque (classes desbalanceadas)
combined_data['attack_type'].value_counts()

16    15600
14     9861
25     1418
4      1240
27     1156
11      996
35      951
7       851
20      743
0       737
21      685
1       555
15      374
29      331
24      309
10      293
32      200
34      181
28      178
5       133
19       79
2        26
12       20
13       17
23       17
22       15
26       14
39       13
37        9
8         8
6         6
18        4
38        4
3         4
9         3
36        2
33        2
17        2
31        2
30        1
Name: attack_type, dtype: int64

* Random Forest (Floresta Aleatória):
O Random Forest é um algoritmo robusto e versátil que funciona bem em muitos
cenários.
Ele pode lidar com dados desbalanceados, pois constrói várias árvores de decisão e combina suas previsões.
Além disso, o Random Forest é menos propenso a overfitting.

* Boosting Algorithms (Algoritmos de Boosting):
Algoritmos de boosting, como CatBoost e LogitBoost, têm mostrado bom desempenho em dados desbalanceados.
Eles criam um conjunto de modelos fracos e os combinam para obter uma previsão mais precisa.

* Decision Trees (Árvores de Decisão):
As árvores de decisão ponderadas podem ser úteis para dados desbalanceados.
Atribuir um custo maior à classe minoritária e um custo menor à classe majoritária pode melhorar o desempenho.

* Ensemble Methods (Métodos de Conjunto):
Combinação de vários modelos, como Bagging e Boosting, pode ser eficaz.
Esses métodos reduzem a variância e melhoram a generalização.

#### Matriz de correlação para escolha das melhores features

In [ ]:
# matriz de correlação para identificar quais das 43 features são as melhores para o modelo e separação dos dados
corr_matrix = combined_data.corr().abs().sort_values('attack_type')
leastCorrelated = corr_matrix['attack_type'].nsmallest(10)
leastCorrelated = list(leastCorrelated.index)
leastSTD =  combined_data.std().to_frame().nsmallest(5, columns=0)
leastSTD = list(leastSTD.transpose().columns)
featureElimination = set(leastCorrelated + leastSTD)
len(featureElimination)
featureElimination
combined_data_reduced = combined_data.drop(featureElimination,axis=1)
X = combined_data_reduced.drop('attack_type', axis=1)
y = combined_data_reduced.loc[:,['attack_type']]

#### Modelos e Avaliações

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42) # Treinando modelo

In [ ]:
#Gaussian
param_grid_gaussian = {
    'priors': [None],
    'var_smoothing': [1e-9, 1e-8, 1e-7]
}


#RandomForest
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

#KNN
param_grid_knn = {
    'n_neighbors': [3, 5, 7]
}

#DecisionTree
param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 5],
    'min_samples_split': [2, 4],
    'min_samples_leaf': [1, 2]
}


# Inicializando os modelos em um dicionário
models = {
    "Naive Bayes": GridSearchCV(GaussianNB(), param_grid_gaussian, cv=5),
    "Random Forest": GridSearchCV(RandomForestClassifier(), param_grid_rf, cv=5),
    "Knn": GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5),
    "DecisionTree": GridSearchCV(DecisionTreeClassifier(), param_grid_dt, cv=5, scoring='recall', n_jobs = -1),
}

for model_name, model in models.items():
    model.fit(X_train, y_train)


In [ ]:
# Escolhendo os melhores parametros
melhores_parametros_gaussian = models["Naive Bayes"].best_params_
melhores_parametros_rf = models["Random Forest"].best_params_
melhores_parametros_knn = models["Knn"].best_params_
melhores_parametros_dt = models["DecisionTree"].best_params_

In [ ]:
# Inicializando os modelos em um dicionário
models = {
    "Naive Bayes": GaussianNB(priors= melhores_parametros_gaussian['priors'], var_smoothing= melhores_parametros_gaussian['var_smoothing']),
    "Random Forest": RandomForestClassifier(n_estimators= melhores_parametros_rf['n_estimators'],max_depth= melhores_parametros_rf['max_depth'],
                                            min_samples_split= melhores_parametros_rf['min_samples_split'],min_samples_leaf= melhores_parametros_rf['min_samples_leaf']),
    "Knn": KNeighborsClassifier(n_neighbors= melhores_parametros_knn['n_neighbors']),
    "DecisionTree": DecisionTreeClassifier(criterion= melhores_parametros_dt['criterion'],max_depth= melhores_parametros_dt['max_depth'],
                                           min_samples_split= melhores_parametros_dt['min_samples_split'],min_samples_leaf= melhores_parametros_dt['min_samples_leaf']),
}


In [ ]:
from sklearn.metrics import classification_report

# Avaliando os modelos de Classificação
for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(classification_report(y_test.to_numpy().ravel(), y_pred))

# conclusão
# naive bayes: apresentou algumas classes com 0 na métrica, classes que estavam com poucas amostras indicando
# que os dados devem ser balanceados para usar esse tipo de algoritmo, embora a acurácia final tenha ficado em 61%.

# random forest: apresentou valores consideraveis de métrica, mesmo com poucas amostras, o que indica que é um algoritmo
# que pode ser utilizado em classes desbalanceadas, a acurácia final foi a mais alta de todos os modelos.

# knn:  apresentou algumas classes com 0 na métrica, classes que estavam com poucas amostras indicando
# que os dados devem ser balanceados para usar esse tipo de algoritmo, embora a acurácia final tenha ficado em 94%.

# decision tree: apresentou o pior resultado, com muitas classes com 0 na métrica mesmo sendo um algoritmo que pode ser
# usado com classes desbalanceadas, mas para classes que estavam com
# um numero alto de amostras, o algoritmo foi bem e isso deixou com acurácia total de 69%.

              precision    recall  f1-score   support

           0       0.99      0.97      0.98       149
           1       0.33      0.96      0.50       130
           2       0.17      0.25      0.20         4
           3       0.00      0.00      0.00         0
           4       0.25      0.88      0.39       250
           5       0.00      0.00      0.00        23
           6       0.00      0.00      0.00         0
           7       0.36      0.83      0.50       159
           8       0.02      0.33      0.03         3
          10       0.94      0.97      0.95        63
          11       0.55      0.67      0.60       187
          12       0.00      0.00      0.00         3
          13       0.00      0.00      0.00         3
          14       0.80      0.98      0.88      1942
          15       0.05      0.12      0.07        82
          16       1.00      0.34      0.51      3130
          17       0.00      0.00      0.00         1
          18       0.00    

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers

y = tf.keras.utils.to_categorical(y)

# Modelo da rede neural
model = Sequential()
model.add(layers.BatchNormalization())
model.add(Dense(14, activation='relu',input_dim=len(X_train.columns)))
model.add(Dense(25, activation='relu'))
model.add(Dense(40, activation='softmax'))


# Compilando o modelo
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Treinando o modelo
model.fit(X_train, y_train, epochs=10)

# Avaliando a acurácia do modelo
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Acurácia do modelo: {test_acc:.2f}')

Epoch 1/10
926/926 [==============================] - 3s 2ms/step - loss: 0.8795 - accuracy: 0.7768
Epoch 2/10
926/926 [==============================] - 2s 2ms/step - loss: 0.3501 - accuracy: 0.8906
Epoch 3/10
926/926 [==============================] - 2s 2ms/step - loss: 0.3046 - accuracy: 0.9026
Epoch 4/10
926/926 [==============================] - 3s 3ms/step - loss: 0.2798 - accuracy: 0.9094
Epoch 5/10
926/926 [==============================] - 3s 3ms/step - loss: 0.2683 - accuracy: 0.9123
Epoch 6/10
926/926 [==============================] - 2s 2ms/step - loss: 0.2595 - accuracy: 0.9149
Epoch 7/10
926/926 [==============================] - 4s 4ms/step - loss: 0.2476 - accuracy: 0.9178
Epoch 8/10
926/926 [==============================] - 6s 7ms/step - loss: 0.2409 - accuracy: 0.9210
Epoch 9/10
926/926 [==============================] - 7s 7ms/step - loss: 0.2376 - accuracy: 0.9221
Epoch 10/10
232/232 [==============================] - 1s 4ms/step - loss: 0.2300 - accuracy: 0.9308

In [ ]:
predictions = model.predict(X_test)

232/232 [==============================] - 1s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

# Avaliando o modelo de rede neural

print(classification_report(y_test.to_numpy().ravel(), predictions.argmax(axis=1)))

# conclusão: algumas classes com poucas amostras ficaram com valores 0 nas métricas na rede neural, mas as métricas
# gerais do modelo foram boas, mas não está adequado se utilizar antes de balancear as amostras.

              precision    recall  f1-score   support

           0       0.98      0.99      0.98       149
           1       0.97      0.78      0.87       130
           2       0.00      0.00      0.00         4
           4       0.96      0.87      0.92       250
           5       0.79      0.83      0.81        23
           7       0.77      0.96      0.86       159
           8       0.00      0.00      0.00         3
          10       0.78      0.86      0.82        63
          11       0.93      0.89      0.91       187
          12       0.00      0.00      0.00         3
          13       0.00      0.00      0.00         3
          14       1.00      1.00      1.00      1942
          15       0.86      0.70      0.77        82
          16       0.95      0.94      0.94      3130
          17       0.00      0.00      0.00         1
          18       0.00      0.00      0.00         1
          19       0.80      0.89      0.84        18
          20       0.92    

#### Balanceamento de classes e novos modelos

In [ ]:
# Balanceando classes para melhorar modelos utilizando SMOTE

from imblearn.over_sampling import SMOTE

# Encontrando as classes com value_counts igual a 1
df_smote = combined_data_reduced['attack_type'].value_counts()[combined_data_reduced['attack_type'].value_counts() >= 100].index

# Removendo as linhas com essas classes
df_smote_filtrado = combined_data_reduced[combined_data_reduced['attack_type'].isin(df_smote)]

# Dados para treinamento
X = df_smote_filtrado.drop('attack_type', axis=1)
y = df_smote_filtrado.loc[:,['attack_type']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1,random_state=42) # Treinando modelo

# Aplicar o SMOTE apenas ao conjunto de treinamento
smote = SMOTE(sampling_strategy='auto',random_state=42, k_neighbors=1, n_jobs=1)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


In [ ]:
# Inicializando o modelo de menor acurácia em um dicionário
models = {
    "Naive Bayes": GaussianNB(priors= melhores_parametros_gaussian['priors'], var_smoothing= melhores_parametros_gaussian['var_smoothing']),
}

for model_name, model in models.items():
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test)
    print(classification_report(y_test.to_numpy().ravel(), y_pred))

# conclusão
# naive bayes: apresentou acurácia geral muito próxima do modelo antes do balanceamento de classes, porém o número
# de classes com 0 na métrica caiu de 17 para 4 classes, o que mostra que o impacto do balanceamento foi positivo.


              precision    recall  f1-score   support

           0       0.95      0.97      0.96        77
           1       0.22      1.00      0.36        51
           4       0.22      0.84      0.35       122
           5       0.00      0.00      0.00        14
           7       0.00      0.00      0.00        86
          10       0.90      0.95      0.92        19
          11       0.45      0.73      0.56        99
          14       0.88      0.95      0.91       999
          15       0.10      1.00      0.17        37
          16       1.00      0.34      0.51      1575
          20       1.00      0.34      0.50        80
          21       0.94      1.00      0.97        67
          24       0.00      0.00      0.00        33
          25       0.74      0.43      0.54       135
          27       0.96      0.98      0.97       109
          28       0.13      0.08      0.10        25
          29       0.94      0.97      0.95        32
          32       0.24    

In [ ]:
# Inicializando modelo de rede neural
y = tf.keras.utils.to_categorical(y)

# Modelo da rede neural
model = Sequential()
model.add(layers.BatchNormalization())
model.add(Dense(14, activation='relu',input_dim=len(X_train.columns)))
model.add(Dense(25, activation='relu'))
model.add(Dense(40, activation='softmax'))


# Compilando o modelo
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Treinando o modelo
model.fit(X_train_smote, y_train_smote, epochs=10)

# Avaliando a acurácia do modelo
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Acurácia do modelo: {test_acc:.2f}')

Epoch 1/10
8766/8766 [==============================] - 31s 3ms/step - loss: 0.4233 - accuracy: 0.8448
Epoch 2/10
8766/8766 [==============================] - 29s 3ms/step - loss: 0.2653 - accuracy: 0.8893
Epoch 3/10
8766/8766 [==============================] - 26s 3ms/step - loss: 0.2456 - accuracy: 0.8972
Epoch 4/10
8766/8766 [==============================] - 32s 4ms/step - loss: 0.2339 - accuracy: 0.9021
Epoch 5/10
8766/8766 [==============================] - 34s 4ms/step - loss: 0.2292 - accuracy: 0.9033
Epoch 6/10
8766/8766 [==============================] - 20s 2ms/step - loss: 0.2244 - accuracy: 0.9050
Epoch 7/10
8766/8766 [==============================] - 19s 2ms/step - loss: 0.2187 - accuracy: 0.9088
Epoch 8/10
8766/8766 [==============================] - 20s 2ms/step - loss: 0.2178 - accuracy: 0.9083
Epoch 9/10
8766/8766 [==============================] - 20s 2ms/step - loss: 0.2162 - accuracy: 0.9090
Epoch 10/10
115/115 [==============================] - 0s 1ms/step - loss

In [ ]:
predictions = model.predict(X_test)

# Avaliando o modelo de rede neural

print(classification_report(y_test.to_numpy().ravel(), predictions.argmax(axis=1)))

# conclusão: com a acurácia próximo ao valor antes do balanceamento, o modelo apresentou apenas uma classe com valor 0 de métrica,
# o que mostra que o balanceamento favoreceu para que as outras classes tivessem melhor desempenho. Um número maior de épocas ou a parametrização do batch_size
# podem ser dois pontos que se ajustados podem melhorar a métrica final do modelo.

115/115 [==============================] - 0s 1ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        77
           1       0.94      0.96      0.95        51
           4       0.95      0.99      0.97       122
           5       0.78      1.00      0.88        14
           7       0.84      0.93      0.88        86
          10       0.45      1.00      0.62        19
          11       0.85      0.95      0.90        99
          14       1.00      0.99      0.99       999
          15       0.74      0.95      0.83        37
          16       1.00      0.84      0.91      1575
          20       0.91      0.97      0.94        80
          21       0.97      0.99      0.98        67
          24       0.35      0.85      0.49        33
          25       0.69      0.64      0.66       135
          27       1.00      0.99      1.00       109
          28       0.26      1.00      0.41        25
          29       0.00   